In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import tempfile
import os

# =========================
# 1. PATHS INSTELLEN
# =========================

BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

AUDIO_ROOT = BASE_DIR / "TAME" / "wiener_gaussian_noise_perturbations" / "low_gaussian_noise"

OUTPUT_FEATURES_CSV = BASE_DIR / "opensmile_low_gaussian_features.csv"
OUTPUT_FAILED_CSV = BASE_DIR / "opensmile_low_gaussian_failed_files.csv"

# check
print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())
print("Audio root exists:", AUDIO_ROOT.exists())

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")

if not AUDIO_ROOT.exists():
    raise FileNotFoundError(f"Audio root not found:\n{AUDIO_ROOT}")

#WAV files
wav_files = sorted(AUDIO_ROOT.rglob("*.wav"))
print(f"Number of wav. files found: {len(wav_files)}")

if len(wav_files) == 0:
    raise ValueError("no .wav files found.")

# Run OpenSMILE
all_rows = []
failed_files = []

for i, wav_file in enumerate(wav_files, start=1):
    print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

    participant_id = wav_file.parent.name
    filename = wav_file.name

    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
        temp_output_csv = Path(tmp.name)

    try:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)

        cmd = [
            str(OPENSMILE_EXE),
            "-C", str(CONFIG_FILE),
            "-I", str(wav_file),
            "-csvoutput", str(temp_output_csv),
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "openSMILE return code != 0",
                "stderr": result.stderr
            })
            continue

        if not temp_output_csv.exists():
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "no output made",
                "stderr": result.stderr
            })
            continue

        try:
            feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error in csv: {e}",
                "stderr": result.stderr
            })
            continue

        if feature_df.empty:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "Output csv is leeg",
                "stderr": result.stderr
            })
            continue

        #adding information 
        row = feature_df.iloc[0].to_dict()
        row["name"] = filename
        row["participant_id"] = participant_id
        row["filename"] = filename
        row["file_path"] = str(wav_file)

        all_rows.append(row)

    except Exception as e:
        failed_files.append({
            "participant_id": participant_id,
            "filename": filename,
            "file_path": str(wav_file),
            "reason": f"error {e}",
            "stderr": ""
        })

    finally:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)


#Save
features_df = pd.DataFrame(all_rows)

# Kolommen netjes ordenen
preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
features_df = features_df[[col for col in preferred_front_cols if col in features_df.columns] + remaining_cols]

features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("\nKlaar.")
print("number of errors in features:", len(features_df))
print("numer of errors in the files:", len(failed_files))
print("Features saves in :")
print(OUTPUT_FEATURES_CSV)

if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_df.to_csv(OUTPUT_FAILED_CSV, index=False)
    print("\n errors saved:")
    print(OUTPUT_FAILED_CSV)


# Results
print("\nShape features dataframe:", features_df.shape)
display(features_df.head())

SMILExtract exists: True
Config exists: True
Audio root exists: True
Aantal wav files gevonden: 1941
[1/1941] Processing: p10085.LC.1.161.wav
[2/1941] Processing: p10085.LC.10.155.wav
[3/1941] Processing: p10085.LC.11.99999.wav
[4/1941] Processing: p10085.LC.12.59.wav
[5/1941] Processing: p10085.LC.14.45.wav
[6/1941] Processing: p10085.LC.16.99999.wav
[7/1941] Processing: p10085.LC.18.41.wav
[8/1941] Processing: p10085.LC.2.106.wav
[9/1941] Processing: p10085.LC.22.22.wav
[10/1941] Processing: p10085.LC.23.156.wav
[11/1941] Processing: p10085.LC.24.172.wav
[12/1941] Processing: p10085.LC.25.112.wav
[13/1941] Processing: p10085.LC.26.99999.wav
[14/1941] Processing: p10085.LC.28.0.wav
[15/1941] Processing: p10085.LC.29.188.wav
[16/1941] Processing: p10085.LC.3.60.wav
[17/1941] Processing: p10085.LC.30.132.wav
[18/1941] Processing: p10085.LC.31.99999.wav
[19/1941] Processing: p10085.LC.32.120.wav
[20/1941] Processing: p10085.LC.33.6.wav
[21/1941] Processing: p10085.LC.34.122.wav
[22/1941]

,participant_id,filename,file_path,name,frameTime,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,p10085,p10085.LC.1.161.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.1.161.wav,0.0,30.90199,0.256474,30.69185,33.91224,35.34587,...,-0.049021,-0.001563,0.083841,3.759399,3.065134,0.088750,0.044564,0.192222,0.162123,-36.63445
1,p10085,p10085.LC.10.155.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.10.155.wav,0.0,34.39622,0.150116,33.80017,35.04958,36.49067,...,-0.042634,0.001876,0.078781,2.857143,2.916667,0.064286,0.041008,0.258571,0.265245,-37.60513
2,p10085,p10085.LC.11.99999.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.11.99999.wav,0.0,31.86741,0.250341,33.06512,35.15099,36.21114,...,-0.044217,-0.001073,0.082083,4.687500,4.126984,0.070769,0.052837,0.163333,0.267592,-37.42188
3,p10085,p10085.LC.12.59.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.12.59.wav,0.0,35.56821,0.142905,35.00095,35.37488,38.83699,...,-0.048745,0.003391,0.080458,3.773585,3.381643,0.081429,0.039795,0.194286,0.175650,-35.83936
4,p10085,p10085.LC.14.45.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.14.45.wav,0.0,34.36342,0.038541,33.35033,34.00417,35.33556,...,-0.052997,0.001773,0.078584,3.083700,2.252252,0.138000,0.091957,0.236667,0.375751,-36.32655
